In [1]:
tabla ='qtiod10'
col_tabla = 'solopefec'
essi='essi_dat_cqx004_etl'
col_essi='sol_fec'

In [2]:
from decouple import config
from sqlalchemy import create_engine,text
import pandas as pd
from datetime import datetime, timedelta
import time 
import oracledb
import sys
import psycopg2
import numpy as np

In [3]:
inicioTiempo = time.time()
inicioProceso=time.time()
now_inicio = datetime.now()

In [4]:
#CONEXIONES
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="essi_etl"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

In [5]:
# fecha = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_mod=13", con=connection2)
# fecha= fecha.iloc[0, 0]
# print(fecha)
now = datetime.now()
# query=f"UPDATE etl_act SET fec_act ='{now}' WHERE id_mod=13"
# c1= text(query)
# connection2.execute(c1)

#INICIO DEL ESSI

In [6]:
fecha = '01/05/23'

In [7]:
oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine0 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection0 = engine0.connect()

query0 = f"""
select
d1.INFOPEORICENASICOD,
d1.INFOPECENASICOD,
d1.INFOPESOLOPENUM,
d1.INFOPESECNUM,
d1.CONDDIAGCOD,
d1.INFOPEDIAGCOD,
d1.TIPODIAGCOD,
d1.INFOPEDIAGORD,

a1.solopefec as solopefec,  
a1.SOLOPEACTMEDNUM,
a1.SOLOPEBUSPACSECNUM
from {tabla} d1
left outer join qtsop10 a1 on a1.SOLOPEORICENASICOD = d1.INFOPEORICENASICOD
and a1.SOLOPECENASICOD    = d1.INFOPECENASICOD
and a1.SOLOPENUM    = d1.INFOPESOLOPENUM

where a1.{col_tabla}>='{fecha}'
"""

base1 = pd.read_sql_query(query0,con=connection0)


connection0.close()

In [8]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50840 entries, 0 to 50839
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   infopeoricenasicod  50840 non-null  object        
 1   infopecenasicod     50840 non-null  object        
 2   infopesolopenum     50840 non-null  int64         
 3   infopesecnum        50840 non-null  int64         
 4   conddiagcod         50840 non-null  object        
 5   infopediagcod       50840 non-null  object        
 6   tipodiagcod         50840 non-null  object        
 7   infopediagord       50840 non-null  int64         
 8   solopefec           50840 non-null  datetime64[ns]
 9   solopeactmednum     50840 non-null  int64         
 10  solopebuspacsecnum  50840 non-null  int64         
dtypes: datetime64[ns](1), int64(5), object(5)
memory usage: 4.3+ MB


In [9]:
base1.columns

Index(['infopeoricenasicod', 'infopecenasicod', 'infopesolopenum',
       'infopesecnum', 'conddiagcod', 'infopediagcod', 'tipodiagcod',
       'infopediagord', 'solopefec', 'solopeactmednum', 'solopebuspacsecnum'],
      dtype='object')

In [10]:
base1 = base1.rename(columns={
    'infopeoricenasicod': 'ori_cas',
    'infopecenasicod': 'cod_cas',
    'infopesolopenum': 'sol_num',
    'infopesecnum': 'sec_num',
    'conddiagcod': 'con_dia',
    'infopediagcod': 'cod_cie',
    'tipodiagcod': 'tip_dia',
    'infopediagord': 'ord_dia',
    'solopefec': 'sol_fec',
    'solopeactmednum': 'act_med',
    'solopebuspacsecnum': 'pac_sec'
})

In [11]:
base1.shape

(50840, 11)

In [12]:
base2=pd.read_sql_query(f"SELECT * FROM {essi} LIMIT 10", con=connection1)

In [13]:
base2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 17 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   ori_cas  10 non-null     object 
 1   cod_cas  10 non-null     object 
 2   des_cas  10 non-null     object 
 3   cod_red  10 non-null     object 
 4   des_red  10 non-null     object 
 5   sol_num  10 non-null     float64
 6   sec_num  10 non-null     float64
 7   con_dia  10 non-null     object 
 8   des_con  10 non-null     object 
 9   cod_cie  10 non-null     object 
 10  des_cie  10 non-null     object 
 11  tip_dia  10 non-null     object 
 12  des_tip  10 non-null     object 
 13  ord_dia  10 non-null     float64
 14  sol_fec  10 non-null     object 
 15  act_med  10 non-null     float64
 16  pac_sec  10 non-null     float64
dtypes: float64(5), object(12)
memory usage: 1.5+ KB


#TRAEMOS TODOS LOS MAESTROS

In [14]:
cas = pd.read_sql_query(f"SELECT id_red,cod_cas,des_cas FROM dim_cas where id_red is not null", con=connection2)
cas = cas.drop_duplicates(subset=['cod_cas'])
cas=cas.dropna()
red = pd.read_sql_query(f"SELECT id_red,cod_red,des_red FROM dim_red", con=connection2)
cas_red=pd.merge(cas,red,how='left',on='id_red')
#id_red,cod_cas,des_cas,cod_red,des_red

condiag=pd.read_sql_query(f"SELECT cod_con,des_con FROM dim_condiag", con=connection2)
condiag['cod_con']=condiag['cod_con'].str.strip()

cie10=pd.read_sql_query(f"SELECT cod_cie,des_cie FROM dim_cie10", con=connection2)
cie10['cod_cie']=cie10['cod_cie'].str.strip()

tipdx=pd.read_sql_query(f"SELECT cod_tdx,des_tdx FROM dim_tipdx", con=connection2)
tipdx['cod_tdx']=tipdx['cod_tdx'].str.strip()

In [15]:
a=base1.copy()

In [16]:
# base1=a

In [17]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50840 entries, 0 to 50839
Data columns (total 11 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   ori_cas  50840 non-null  object        
 1   cod_cas  50840 non-null  object        
 2   sol_num  50840 non-null  int64         
 3   sec_num  50840 non-null  int64         
 4   con_dia  50840 non-null  object        
 5   cod_cie  50840 non-null  object        
 6   tip_dia  50840 non-null  object        
 7   ord_dia  50840 non-null  int64         
 8   sol_fec  50840 non-null  datetime64[ns]
 9   act_med  50840 non-null  int64         
 10  pac_sec  50840 non-null  int64         
dtypes: datetime64[ns](1), int64(5), object(5)
memory usage: 4.3+ MB


In [18]:
base1=pd.merge(base1,cas_red,how='left',on='cod_cas')
base1=base1.drop("id_red", axis=1)
base1.shape

(50840, 14)

In [19]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 50840 entries, 0 to 50839
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   ori_cas  50840 non-null  object        
 1   cod_cas  50840 non-null  object        
 2   sol_num  50840 non-null  int64         
 3   sec_num  50840 non-null  int64         
 4   con_dia  50840 non-null  object        
 5   cod_cie  50840 non-null  object        
 6   tip_dia  50840 non-null  object        
 7   ord_dia  50840 non-null  int64         
 8   sol_fec  50840 non-null  datetime64[ns]
 9   act_med  50840 non-null  int64         
 10  pac_sec  50840 non-null  int64         
 11  des_cas  50840 non-null  object        
 12  cod_red  50840 non-null  object        
 13  des_red  50840 non-null  object        
dtypes: datetime64[ns](1), int64(5), object(8)
memory usage: 5.8+ MB


In [20]:
#des_con
base1['con_dia']=base1['con_dia'].str.strip()
base1=pd.merge(base1,condiag,how='left',left_on='con_dia',right_on='cod_con')
base1 = base1.drop("cod_con", axis=1)
base1.shape

(50840, 15)

In [21]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 50840 entries, 0 to 50839
Data columns (total 15 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   ori_cas  50840 non-null  object        
 1   cod_cas  50840 non-null  object        
 2   sol_num  50840 non-null  int64         
 3   sec_num  50840 non-null  int64         
 4   con_dia  50840 non-null  object        
 5   cod_cie  50840 non-null  object        
 6   tip_dia  50840 non-null  object        
 7   ord_dia  50840 non-null  int64         
 8   sol_fec  50840 non-null  datetime64[ns]
 9   act_med  50840 non-null  int64         
 10  pac_sec  50840 non-null  int64         
 11  des_cas  50840 non-null  object        
 12  cod_red  50840 non-null  object        
 13  des_red  50840 non-null  object        
 14  des_con  50840 non-null  object        
dtypes: datetime64[ns](1), int64(5), object(9)
memory usage: 6.2+ MB


In [22]:
#des_cie
base1['cod_cie']=base1['cod_cie'].str.strip()
base1=pd.merge(base1,cie10,how='left',on='cod_cie')
base1.shape

(50840, 16)

In [23]:
#des_tip
base1['tip_dia']=base1['tip_dia'].str.strip()
base1=pd.merge(base1,tipdx,how='left',left_on='tip_dia',right_on='cod_tdx')
base1=base1.rename(columns={"des_tdx": "des_tip"})
base1 = base1.drop("cod_tdx", axis=1)
base1.shape

(50840, 17)

In [24]:
base2.columns

Index(['ori_cas', 'cod_cas', 'des_cas', 'cod_red', 'des_red', 'sol_num',
       'sec_num', 'con_dia', 'des_con', 'cod_cie', 'des_cie', 'tip_dia',
       'des_tip', 'ord_dia', 'sol_fec', 'act_med', 'pac_sec'],
      dtype='object')

In [25]:
df1_columns = set(base1.columns)
df2_columns = set(base2.columns) 
different_columns = df2_columns - df1_columns
different_columns

set()

In [26]:
df1_columns = set(base1.columns)
df2_columns = set(base2.columns) 
different_columns = df1_columns - df2_columns
different_columns

set()

In [27]:
comunes = set(base1.columns).intersection(set(base2.columns)) 
base = base1[list(comunes)]

In [28]:
base.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 50840 entries, 0 to 50839
Data columns (total 17 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   sec_num  50840 non-null  int64         
 1   pac_sec  50840 non-null  int64         
 2   sol_num  50840 non-null  int64         
 3   des_tip  50840 non-null  object        
 4   cod_red  50840 non-null  object        
 5   des_cie  50840 non-null  object        
 6   ord_dia  50840 non-null  int64         
 7   des_cas  50840 non-null  object        
 8   des_con  50840 non-null  object        
 9   tip_dia  50840 non-null  object        
 10  sol_fec  50840 non-null  datetime64[ns]
 11  act_med  50840 non-null  int64         
 12  ori_cas  50840 non-null  object        
 13  cod_cie  50840 non-null  object        
 14  cod_cas  50840 non-null  object        
 15  des_red  50840 non-null  object        
 16  con_dia  50840 non-null  object        
dtypes: datetime64[ns](1), int64(5),

In [29]:
base.head()

,sec_num,pac_sec,sol_num,des_tip,cod_red,des_cie,ord_dia,des_cas,des_con,tip_dia,sol_fec,act_med,ori_cas,cod_cie,cod_cas,des_red,con_dia
0,1,17437280,5182,PRESUNTIVO,07,ESTERILIZACION ...,1,H.III SUAREZ-ANGAMOS,SALIDA,1,2029-07-10,6799154,1,Z30.2,006,RED PRESTACIONAL REBAGLIATI ...,2
1,1,928443,63927,PRESUNTIVO,07,"TUMOR BENIGNO LIPOMATOSO, DE SITIO NO ESPECIFI...",1,H.N. EDGARDO REBAGLIATI MARTINS,SALIDA,1,2023-06-02,11250874,1,D17.9,001,RED PRESTACIONAL REBAGLIATI ...,2
2,1,13754960,13541,PRESUNTIVO,12,CALCULO DEL URETER ...,1,H.II CAJAMARCA,SALIDA,1,2025-08-05,1504924,1,N20.1,191,RED ASISTENCIAL CAJAMARCA ...,2
3,1,1281848,72546,PRESUNTIVO,07,LINFOMA NO HODGKIN DE CELULAS PEQUEÑAS (DIFUSO...,1,H.N. EDGARDO REBAGLIATI MARTINS,SALIDA,1,2023-09-02,11039911,1,C83.0,001,RED PRESTACIONAL REBAGLIATI ...,2
4,1,3961718,60105,PRESUNTIVO,05,"HERNIA ABDOMINAL NO ESPECIFICADA, SIN OBSTRUCC...",1,H.N. ALBERTO SABOGAL SOLOGUREN,SALIDA,1,2023-10-17,9341987,1,K46.9,005,RED PRESTACIONAL SABOGAL ...,2


In [30]:
base.columns

Index(['sec_num', 'pac_sec', 'sol_num', 'des_tip', 'cod_red', 'des_cie',
       'ord_dia', 'des_cas', 'des_con', 'tip_dia', 'sol_fec', 'act_med',
       'ori_cas', 'cod_cie', 'cod_cas', 'des_red', 'con_dia'],
      dtype='object')

In [31]:
borrando=f"DELETE FROM {essi} WHERE {col_essi} >='{fecha}'"
borrado = connection1.execute(borrando)

In [32]:
base.to_sql(name=f'{essi}', con=connection1, if_exists='append', index=False,chunksize=10000)

5840

In [33]:
finproceso=time.time()
tiempoproceso=finproceso - inicioProceso
tiempoproceso=round(tiempoproceso,3)
print("Proceso 1.2 completado satisfactoriamente en " + str(tiempoproceso)+" segundos")

Proceso 1.2 completado satisfactoriamente en 18.525 segundos


In [34]:
connection1.close()
connection2.close()
connection3.close()

In [35]:
engine1.dispose()
engine2.dispose()
engine3.dispose()